# create database for later data analysis using sqlalchemy
## this will only create data for water, soil, sediment, not for air

Run this script once after model_data_extractor work is finished

REVAMP with intervall mapping for half life

In [3]:
import sys
from typing import List
import numpy as np
import pandas as pd
from pandas import read_csv
from pathlib import Path

notebookdir = Path.cwd().parent
sys.path.append(str(notebookdir))  # this way we can import src modules even in different subfolders

import sqlalchemy as sa
from sqlalchemy.orm import sessionmaker
from src.db_schema import *
from src.rdkit_tools import smiles_standardizer_msc

In [4]:
# set directories and filenames, load data, create db if not exists
working_dir = Path.cwd().parent
data_dir = working_dir / "processed_data"

# create SQLAlchemy engine and session
database_file = data_dir / "vega_t_half_soil_water_sediment.db"
engine = sa.create_engine(f"sqlite:///{database_file}")
Session = sessionmaker(bind=engine)
session = Session()

# drop all tables and recreate from scratch (full rebuild)
Base.metadata.drop_all(engine)
Base.metadata.create_all(engine)

soil_file = data_dir / "vega_soil_t_half_data.csv"
water_file = data_dir / "vega_water_t_half_data.csv"
sediment_file = data_dir / "vega_sediment_t_half_data.csv"
soil_data_all = read_csv(soil_file, index_col=0, header=0, sep="\t")
water_data_all = read_csv(water_file, index_col=0, header=0, sep="\t")
sediment_data_all = read_csv(sediment_file, index_col=0, header=0, sep="\t")

# insert into df counter of format "table_name"+<counter> with counter being 4 digits to each table called "reference"
soil_data_all.insert(0, "reference", ["soil_" + str(i).zfill(4) for i in range(len(soil_data_all))])
water_data_all.insert(0, "reference", ["water_" + str(i).zfill(4) for i in range(len(water_data_all))])
sediment_data_all.insert(0, "reference", ["sediment_" + str(i).zfill(4) for i in range(len(sediment_data_all))])

# simplify column names for half-life to T_half_days
soil_data_all = soil_data_all.rename(columns={"T_half_soil_days": "T_half_days"})
water_data_all = water_data_all.rename(columns={"T_half_water_days": "T_half_days"})
sediment_data_all = sediment_data_all.rename(columns={"T_half_sediment_days": "T_half_days"})

In [5]:
# get all "Canonical_smiles", created Canonical SMILES to ensure all are actually canonical
# this should have been done during earlier data processing, but just to be sure and then not have any issues later on
data_frames = {
    "soil": soil_data_all,
    "sediment": sediment_data_all,
    "water": water_data_all,
}

for df in data_frames.values():
    unique_smiles: List[str] = df["Canonical_smiles"].unique().tolist()
    canonical_smiles = {}
    for smi in unique_smiles:
        canonical_smiles[smi] = smiles_standardizer_msc(smi)[0]

    df["Canonical_smiles"] = df["Canonical_smiles"].map(canonical_smiles)
    df.dropna(subset=["Canonical_smiles"], inplace=True)
    df.reset_index(drop=True, inplace=True)

In [6]:
# --- Interval mapping for T_half_days ---
# Load the vega interval mapping table (class centres only; bounds are derived below)
interval_map_file = Path.cwd() / "vega_interval_mapping.csv"
interval_map = read_csv(interval_map_file)

# Sentinel value used in place of Infinity for the upper bound of the last (open-ended) bin
INFINITY_SENTINEL = 9_999_999.0

# Derive bin bounds from class centres using geometric midpoints (same spacing as hsbd mapping).
# lower_bound of bin i = geometric mean of class centre i-1 and centre i  (first bin: half of centre 0)
# upper_bound of bin i = geometric mean of class centre i and centre i+1  (last bin: INFINITY_SENTINEL)
centres = interval_map["Half_life_days"].values  # shape (9,)
lower_bounds = np.empty(len(centres))
upper_bounds = np.empty(len(centres))

lower_bounds[0] = centres[0] / 2.0  # half of first centre
for i in range(1, len(centres)):
    lower_bounds[i] = np.sqrt(centres[i - 1] * centres[i])  # geometric midpoint

for i in range(len(centres) - 1):
    upper_bounds[i] = lower_bounds[i + 1]  # upper of bin i = lower of bin i+1
upper_bounds[-1] = INFINITY_SENTINEL  # last bin is open-ended

# log10 bounds (sentinel for the open upper end of the last bin)
log10_lower_bounds = np.log10(lower_bounds)
log10_upper_bounds = np.where(
    upper_bounds < INFINITY_SENTINEL,
    np.log10(upper_bounds),
    INFINITY_SENTINEL,
)

interval_map["lower_bound_days"] = lower_bounds
interval_map["upper_bound_days"] = upper_bounds
interval_map["log10_lower"] = log10_lower_bounds
interval_map["log10_upper"] = log10_upper_bounds

print("Derived interval bounds:")
print(
    interval_map[["Class", "Half_life_days", "lower_bound_days", "upper_bound_days", "log10_lower", "log10_upper"]].to_string(
        index=False
    )
)


def map_t_half_to_interval(t_half_series, interval_map):
    """Map each T_half_days value to the corresponding interval bin.

    Returns a DataFrame with columns:
        T_half_class_days, T_half_lower_bound, T_half_upper_bound,
        T_half_log10_lower, T_half_log10_upper

    Rows with NaN T_half_days get NaN in all output columns.
    The upper bound of the last (open-ended) bin is stored as INFINITY_SENTINEL.
    Bins are defined as [lower_bound_days, upper_bound_days).
    """
    out_cols = [
        "T_half_class_days",
        "T_half_lower_bound",
        "T_half_upper_bound",
        "T_half_log10_lower",
        "T_half_log10_upper",
    ]
    result = pd.DataFrame(np.nan, index=t_half_series.index, columns=out_cols)

    for idx, val in t_half_series.items():
        if pd.isna(val):
            continue  # leaves NaN — row will be dropped before DB insert
        for _, bin_row in interval_map.iterrows():
            if bin_row["lower_bound_days"] <= val < bin_row["upper_bound_days"]:
                result.loc[idx, "T_half_class_days"] = bin_row["Half_life_days"]
                result.loc[idx, "T_half_lower_bound"] = bin_row["lower_bound_days"]
                result.loc[idx, "T_half_upper_bound"] = bin_row["upper_bound_days"]  # sentinel for last bin
                result.loc[idx, "T_half_log10_lower"] = bin_row["log10_lower"]
                result.loc[idx, "T_half_log10_upper"] = bin_row["log10_upper"]  # sentinel for last bin
                break
    return result


# Apply interval mapping to all three compartment DataFrames
for df in [soil_data_all, water_data_all, sediment_data_all]:
    mapped = map_t_half_to_interval(df["T_half_days"], interval_map)
    for col in mapped.columns:
        df[col] = mapped[col]

# Drop rows where T_half_days is NaN (cannot be mapped to any interval)
before = {"soil": len(soil_data_all), "water": len(water_data_all), "sediment": len(sediment_data_all)}
soil_data_all = soil_data_all.dropna(subset=["T_half_days"])
water_data_all = water_data_all.dropna(subset=["T_half_days"])
sediment_data_all = sediment_data_all.dropna(subset=["T_half_days"])

for compartment, df in [("soil", soil_data_all), ("water", water_data_all), ("sediment", sediment_data_all)]:
    dropped = before[compartment] - len(df)
    print(f"{compartment}: {len(df)} rows kept, {dropped} dropped (NaN T_half_days)")

print("\nSample interval mapping (soil, first 5 rows):")
print(
    soil_data_all[
        [
            "reference",
            "T_half_days",
            "T_half_class_days",
            "T_half_lower_bound",
            "T_half_upper_bound",
            "T_half_log10_lower",
            "T_half_log10_upper",
        ]
    ].head()
)

Derived interval bounds:
 Class  Half_life_days  lower_bound_days  upper_bound_days  log10_lower   log10_upper
     1           0.208          0.104000      3.837499e-01    -0.982967 -4.159517e-01
     2           0.708          0.383750      1.273867e+00    -0.415952  1.051239e-01
     3           2.292          1.273867      4.029173e+00     0.105124  6.052159e-01
     4           7.083          4.029173      1.274053e+01     0.605216  1.105188e+00
     5          22.917         12.740530      4.028995e+01     1.105188  1.605197e+00
     6          70.833         40.289947      1.274072e+02     1.605197  2.105194e+00
     7         229.167        127.407167      4.028977e+02     2.105194  2.605195e+00
     8         708.333        402.897690      1.274074e+03     2.605195  3.105194e+00
     9        2291.667       1274.073530      9.999999e+06     3.105194  9.999999e+06
soil: 226 rows kept, 0 dropped (NaN T_half_days)
water: 223 rows kept, 0 dropped (NaN T_half_days)
sediment: 221 ro

In [7]:
# Insert data into database using ORM
session.bulk_save_objects(
    [
        SoilData(
            reference=row["reference"],
            T_half_days=row["T_half_days"],
            T_half_class_days=row["T_half_class_days"],
            T_half_lower_bound=row["T_half_lower_bound"],
            T_half_upper_bound=row["T_half_upper_bound"],
            T_half_log10_lower=row["T_half_log10_lower"],
            T_half_log10_upper=row["T_half_log10_upper"],
            Canonical_smiles=row["Canonical_smiles"],
        )
        for _, row in soil_data_all.iterrows()
    ]
)
session.bulk_save_objects(
    [
        WaterData(
            reference=row["reference"],
            T_half_days=row["T_half_days"],
            T_half_class_days=row["T_half_class_days"],
            T_half_lower_bound=row["T_half_lower_bound"],
            T_half_upper_bound=row["T_half_upper_bound"],
            T_half_log10_lower=row["T_half_log10_lower"],
            T_half_log10_upper=row["T_half_log10_upper"],
            Canonical_smiles=row["Canonical_smiles"],
        )
        for _, row in water_data_all.iterrows()
    ]
)
session.bulk_save_objects(
    [
        SedimentData(
            reference=row["reference"],
            T_half_days=row["T_half_days"],
            T_half_class_days=row["T_half_class_days"],
            T_half_lower_bound=row["T_half_lower_bound"],
            T_half_upper_bound=row["T_half_upper_bound"],
            T_half_log10_lower=row["T_half_log10_lower"],
            T_half_log10_upper=row["T_half_log10_upper"],
            Canonical_smiles=row["Canonical_smiles"],
        )
        for _, row in sediment_data_all.iterrows()
    ]
)
session.commit()

# query SoilData number of total columns
n_col = len(SoilData.__table__.columns)
print(f"Number of columns in soil_data: {n_col}")

Number of columns in soil_data: 379


In [8]:
# test and get all data from db using SQLAlchemy ORM
soil_data = pd.DataFrame(
    session.query(
        SoilData.reference,
        SoilData.T_half_days,
        SoilData.T_half_class_days,
        SoilData.T_half_lower_bound,
        SoilData.T_half_upper_bound,
        SoilData.T_half_log10_lower,
        SoilData.T_half_log10_upper,
        SoilData.Canonical_smiles,
    ).all(),
    columns=[
        "reference",
        "T_half_days",
        "T_half_class_days",
        "T_half_lower_bound",
        "T_half_upper_bound",
        "T_half_log10_lower",
        "T_half_log10_upper",
        "Canonical_smiles",
    ],
)
water_data = pd.DataFrame(
    session.query(
        WaterData.reference,
        WaterData.T_half_days,
        WaterData.T_half_class_days,
        WaterData.T_half_lower_bound,
        WaterData.T_half_upper_bound,
        WaterData.T_half_log10_lower,
        WaterData.T_half_log10_upper,
        WaterData.Canonical_smiles,
    ).all(),
    columns=[
        "reference",
        "T_half_days",
        "T_half_class_days",
        "T_half_lower_bound",
        "T_half_upper_bound",
        "T_half_log10_lower",
        "T_half_log10_upper",
        "Canonical_smiles",
    ],
)
sediment_data = pd.DataFrame(
    session.query(
        SedimentData.reference,
        SedimentData.T_half_days,
        SedimentData.T_half_class_days,
        SedimentData.T_half_lower_bound,
        SedimentData.T_half_upper_bound,
        SedimentData.T_half_log10_lower,
        SedimentData.T_half_log10_upper,
        SedimentData.Canonical_smiles,
    ).all(),
    columns=[
        "reference",
        "T_half_days",
        "T_half_class_days",
        "T_half_lower_bound",
        "T_half_upper_bound",
        "T_half_log10_lower",
        "T_half_log10_upper",
        "Canonical_smiles",
    ],
)

print(f"Soil data entries in DB:     {len(soil_data)}")
print(f"Water data entries in DB:    {len(water_data)}")
print(f"Sediment data entries in DB: {len(sediment_data)}")

print("\nExample entries from soil data (interval columns included):")
print(
    soil_data[
        [
            "reference",
            "T_half_days",
            "T_half_class_days",
            "T_half_lower_bound",
            "T_half_upper_bound",
            "T_half_log10_lower",
            "T_half_log10_upper",
        ]
    ].head()
)

session.close()

Soil data entries in DB:     226
Water data entries in DB:    223
Sediment data entries in DB: 221

Example entries from soil data (interval columns included):
   reference  T_half_days  T_half_class_days  T_half_lower_bound  \
0  soil_0000    70.833337             70.833           40.289947   
1  soil_0001  2291.666942           2291.667         1274.073530   
2  soil_0002     7.083333              7.083            4.029173   
3  soil_0003    70.833337             70.833           40.289947   
4  soil_0004    22.916664             22.917           12.740530   

   T_half_upper_bound  T_half_log10_lower  T_half_log10_upper  
0        1.274072e+02            1.605197        2.105194e+00  
1        9.999999e+06            3.105194        9.999999e+06  
2        1.274053e+01            0.605216        1.105188e+00  
3        1.274072e+02            1.605197        2.105194e+00  
4        4.028995e+01            1.105188        1.605197e+00  
